# Benchmark multi-sources étendu — Collection réelle

Objectif :

- partir d’un export réel de collection ;
- sélectionner un échantillon représentatif par série ;
- utiliser le premier volume disponible par série ;
- limiter les doublons liés aux séries longues ;
- benchmarker Google Books, OpenLibrary, BNF et Nudger sur cet échantillon.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT / "src"))



from collectionlens.api.google_books_api import (
    search_book_by_isbn as google_books_isbn,
)
from collectionlens.api.google_books_api import (
    search_book_by_isbn as google_books_isbn,
)

from collectionlens.api.open_library_api import (
    search_book_by_isbn as openlibrary_isbn,
)

from collectionlens.api.bnf_api import (
    search_book_by_isbn as bnf_isbn,
)

from collectionlens.api.nudger_dataset import (
    load_nudger_dataset,
    build_nudger_search_function,
)

from collectionlens.benchmark.api_benchmark import (
    benchmark_source_in_batches,
)



## 1. Construction du dataset benchmark

In [ ]:
collection_path = PROJECT_ROOT / "data" / "raw" / "collection_export.csv"

df_collection = pd.read_csv(
    collection_path,
    dtype={
        "EAN": str,
        "isbn_clean": str,
    },
    sep=";",
    low_memory=False,
)

df_collection.head()

,Titre de la série,Type,Collection,Catégory,Titre de l'album,EAN,Tome,Date de publication,Editeur,Auteurs,...,Version numérique,A vendre,Numérotation,Prix d'achat,Cote,Etat,Ma note de l'album,Mon avis de l'album,Ma note de la série,Mon avis de la série
0,103ème Escadrille de Chasse,NaN,NaN,Mangas,NaN,9782888906063,NaN,2013-09-21T00:00:00.000Z,Paquet,Takizawa Seiho,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2001 Nights Stories,NaN,NaN,Mangas,2001 Nights Stories,9782344057001,1.0,2023-10-04T00:00:00.000Z,Glénat,Yukinobu Hoshino,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2001 Nights Stories,NaN,NaN,Mangas,2001 Nights Stories,9782344057247,2.0,2023-10-04T00:00:00.000Z,Glénat,Yukinobu Hoshino,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20th Century Boys,album simple N&B,NaN,Mangas,NaN,9782845380790,1.0,2003-12-22T00:00:00.000Z,Panini,Naoki Urasawa,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,20th Century Boys,album simple N&B,NaN,Mangas,NaN,9782845380998,2.0,2003-12-22T00:00:00.000Z,Panini,Naoki Urasawa,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_collection_clean = df_collection.copy()

df_collection_clean = df_collection_clean[
    df_collection_clean["EAN"].notna()
].copy()

df_collection_clean["Tome"] = pd.to_numeric(
    df_collection_clean["Tome"],
    errors="coerce",
)

df_collection_clean["Titre de la série"] = (
    df_collection_clean["Titre de la série"]
    .fillna("")
    .str.strip()
)

In [ ]:
df_first_volume_by_series = (
    df_collection_clean
    .sort_values(
        by=[
            "Titre de la série",
            "Tome",
            "Date de publication",
        ],
        ascending=True,
    )
    .groupby("Titre de la série", as_index=False)
    .first()
)

df_first_volume_by_series[
    [
        "Titre de la série",
        "Type",
        "Catégory",
        "Titre de l'album",
        "Tome",
        "EAN",
        "Editeur",
        "Auteurs",
    ]
].head()

,Titre de la série,Type,Catégory,Titre de l'album,Tome,EAN,Editeur,Auteurs
0,#les memes,album simple,BD,Chronique des âges farouches,1.0,9791038202733,Fluide Glacial,Sylvain Frécon
1,103ème Escadrille de Chasse,None,Mangas,None,NaN,9782888906063,Paquet,Takizawa Seiho
2,2001 Nights Stories,None,Mangas,2001 Nights Stories,1.0,9782344057001,Glénat,Yukinobu Hoshino
3,20th Century Boys,album simple N&B,Mangas,None,1.0,9782845380790,Panini,Naoki Urasawa
4,20th Century Boys - Spin off,None,Mangas,None,NaN,9782809425659,Panini,Naoki Urasawa


In [ ]:
print("Nombre d'albums dans l'export :", len(df_collection))
print("Nombre d'albums avec ISBN :", len(df_collection_clean))
print("Nombre de séries représentées :", len(df_first_volume_by_series))

Nombre d'albums dans l'export : 5651
Nombre d'albums avec ISBN : 5651
Nombre de séries représentées : 1036


## 2. Vérification du dataset benchmark

In [ ]:
df_first_volume_by_series["Type"].value_counts(dropna=False)

Type
album simple              361
None                      354
album simple N&B          259
intégrale complète         26
hors série                  7
Autre                       6
intégrale 2T                5
intégrale 3T                4
intégrale complète N&B      3
Fascicule / Softcover       2
intégrale 2T N&B            2
coffret intégrale           2
intégrale 4T                2
coffret complet             1
coffret intégrale N&B       1
intégrale 5T                1
Name: count, dtype: int64

In [ ]:
df_first_volume_by_series["Catégory"].value_counts(dropna=False)

Catégory
Mangas    759
Comics    169
BD        104
None        4
Name: count, dtype: int64

In [ ]:
(
    df_first_volume_by_series
    .groupby("Catégory")["Type"]
    .apply(lambda x: x.isna().sum())
    .reset_index(name="missing_type")
)

,Catégory,missing_type
0,BD,26
1,Comics,22
2,Mangas,303


In [ ]:
df_first_volume_by_series[
    df_first_volume_by_series["Type"].isna()
][
    [
        "Titre de la série",
        "Catégory",
        "Titre de l'album",
        "Tome",
    ]
].head(20)

,Titre de la série,Catégory,Titre de l'album,Tome
1,103ème Escadrille de Chasse,Mangas,None,NaN
2,2001 Nights Stories,Mangas,2001 Nights Stories,1.0
4,20th Century Boys - Spin off,Mangas,None,NaN
6,24 Histoires d'un Temps Lointain,Mangas,None,NaN
7,25 Histoires d'un Monde en 4 Dimensions,Mangas,None,NaN
9,3 Rue des Mystères et Autres Histoires,Mangas,None,1.0
12,7 Shakespeares,Mangas,None,1.0
13,8 Man Infinity,Mangas,8 Man Infinity,1.0
16,Ace Attorney - Investigations,Mangas,None,1.0
17,Ace Attorney - Phoenix Wright,Mangas,None,1.0


In [ ]:
output_dir = PROJECT_ROOT / "data" / "processed" / "extended_benchmark"
output_dir.mkdir(parents=True, exist_ok=True)

benchmark_input_path = output_dir / "benchmark_first_volume_by_series.csv"

df_first_volume_by_series.to_csv(
    benchmark_input_path,
    index=False,
)

benchmark_input_path

WindowsPath('c:/Users/yoann/Documents/mes projets/CollectionLens V2/data/processed/extended_benchmark/benchmark_first_volume_by_series.csv')

In [ ]:
isbns = (
    df_first_volume_by_series["EAN"]
    .dropna()
    .drop_duplicates()
    .tolist()
)

len(isbns)

1036

L'analyse des valeurs manquantes du champ `Type` montre qu'elles concernent principalement des mangas. Une vérification manuelle indique qu'il s'agit majoritairement de one-shots, d'éditions uniques ou de séries sans numérotation explicite.

Ces valeurs ne sont donc pas considérées comme des anomalies et ne nécessitent pas de correction pour la suite du benchmark.

Le benchmark étendu est construit à partir de l’export réel de collection.

Pour éviter de surreprésenter les séries longues, un seul volume est conservé par série : le premier tome disponible selon le numéro de tome et la date de publication.

Cette stratégie permet d’obtenir un échantillon plus représentatif de la diversité des séries présentes dans la collection.

## 3. Benchmark Google Books

In [ ]:
benchmark_isbns = (
    df_first_volume_by_series["isbn_clean"]
    .dropna()
    .drop_duplicates()
    .tolist()
)

print(f"Nombre d'ISBN benchmark : {len(benchmark_isbns)}")

In [ ]:
from collectionlens.api.google_books_api import (
    search_book_by_isbn as google_books_isbn,
)

df_google_books = benchmark_source_in_batches(
    source_name="google_books",
    search_func=google_books_isbn,
    isbns=benchmark_isbns,
    batch_size=50,
    output_dir=output_dir / "google_books",
    sleep_seconds=1.0,
)

df_google_books.to_csv(
    output_dir / "google_books_results.csv",
    index=False,
)

df_google_books.head()

## 4. Benchmark open library

In [ ]:
from collectionlens.api.open_library_api import (
    search_book_by_isbn as openlibrary_isbn,
)

df_openlibrary = benchmark_source_in_batches(
    source_name="openlibrary",
    search_func=openlibrary_isbn,
    isbns=benchmark_isbns,
    batch_size=50,
    output_dir=output_dir / "openlibrary",
    sleep_seconds=1.0,
)

df_openlibrary.to_csv(
    output_dir / "openlibrary_results.csv",
    index=False,
)

df_openlibrary.head()

## 5. Benchmark BNF

In [ ]:
from collectionlens.api.bnf_api import (
    search_book_by_isbn as bnf_isbn,
)

df_bnf = benchmark_source_in_batches(
    source_name="bnf",
    search_func=bnf_isbn,
    isbns=benchmark_isbns,
    batch_size=50,
    output_dir=output_dir / "bnf",
    sleep_seconds=1.0,
)

df_bnf.to_csv(
    output_dir / "bnf_results.csv",
    index=False,
)

df_bnf.head()

## 6. Benchmark nudger

In [ ]:
nudger_path = PROJECT_ROOT / "data" / "external" / "nudger" / "open4goods-isbn-dataset.csv"

df_nudger_source = load_nudger_dataset(
    csv_path=nudger_path,
    isbn_column="isbn",
)

nudger_isbn = build_nudger_search_function(df_nudger_source)


df_nudger = benchmark_source_in_batches(
    source_name="nudger",
    search_func=nudger_isbn,
    isbns=benchmark_isbns,
    batch_size=100,
    output_dir=output_dir / "nudger",
    sleep_seconds=0.0,
)

df_nudger.to_csv(
    output_dir / "nudger_results.csv",
    index=False,
)

df_nudger.head()